In [0]:
# =============================================================================
# NOTEBOOK  : gold_dim_customer
# PURPOSE   : silver.customers → gold.dim_customer
# LAYER     : Gold (dimension table)
# FREQUENCY : Daily (after silver_customers completes)
# WRITE     : MERGE on customer_id — incremental, customers update over time
#             (loyalty_tier upgrades, total_spend changes, etc.)
# =============================================================================
 
# ── CELL 1: Imports & Config ──────────────────────────────────────────────────
import sys, json
 
_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
PROJECT_ROOT = "/Workspace" + _notebook_path.split("/mini_project_a3t7/")[0] + "/mini_project_a3t7"
 
sys.path.insert(0, PROJECT_ROOT)
 
from utils.audit import start_audit, end_audit, get_last_successful_run_time
from utils.schema_registry import GOLD_DIM_CUSTOMER_SCHEMA
 
from pyspark.sql.functions import current_timestamp, lit, col
from delta.tables import DeltaTable
 
with open(f"{PROJECT_ROOT}/config/config.json") as f:
    cfg = json.load(f)
 
SOURCE_TABLE = cfg["tables"]["silver_customers"]
TARGET_TABLE = cfg["tables"]["gold_dim_customer"]
PIPELINE     = "gold_dim_customer"

In [0]:
# ── Read + Transform + Merge ─────────────────────────────────────────
run_id = start_audit(spark, PROJECT_ROOT, PIPELINE, TARGET_TABLE, SOURCE_TABLE)
 
try:
    # Incremental read — only customers updated since last run
    last_run_time = get_last_successful_run_time(spark, PROJECT_ROOT, PIPELINE)
 
    if last_run_time:
        df = (
            spark.read.table(SOURCE_TABLE)
            .filter(col("processed_at") > lit(last_run_time))
        )
    else:
        df = spark.read.table(SOURCE_TABLE)
 
    rows_read = df.count()
    print(f"[READ] {rows_read} new or updated customer records")
 
    if rows_read == 0:
        print("[SKIP] No new customers to process")
        end_audit(spark, PROJECT_ROOT, run_id, "SUCCESS",
                  rows_read=0, rows_written=0)
        raise SystemExit(0)
 
    df = (
        df
        .withColumn("_gold_processed_at", current_timestamp())
        .withColumn("pipeline_run_id",    lit(run_id))
        .select([f.name for f in GOLD_DIM_CUSTOMER_SCHEMA.fields])
    )
 
    # MERGE — handles both new customers (INSERT) and
    # updated customers (loyalty_tier change, total_spend change, etc.)
    (
        DeltaTable.forName(spark, TARGET_TABLE).alias("t")
        .merge(df.alias("s"), "t.customer_id = s.customer_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
 
    metrics = (
        spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1")
        .select("operationMetrics").collect()[0][0]
    )
    rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
    rows_updated  = int(metrics.get("numTargetRowsUpdated",  0))
 
    print(f"[DONE] inserted={rows_inserted} | updated={rows_updated}")
 
    end_audit(spark, PROJECT_ROOT, run_id, "SUCCESS",
              rows_read=rows_read,
              rows_written=rows_inserted + rows_updated,
              extra={
                  "rows_inserted": str(rows_inserted),
                  "rows_updated":  str(rows_updated)
              })
 
except SystemExit:
    pass
except Exception as e:
    end_audit(spark, PROJECT_ROOT, run_id, "FAILED", error=str(e))
    raise

In [0]:
# ── Verify ───────────────────────────────────────────────────────────
from pyspark.sql.functions import col
 
spark.read.table("azure3_team7_project.platform.pipeline_audit") \
    .filter(col("pipeline_name") == PIPELINE) \
    .orderBy(col("start_time").desc()).limit(1) \
    .select("status", "rows_read", "rows_written", "extra_metadata").show(truncate=False)
 
print("Row count:", spark.read.table(TARGET_TABLE).count())
spark.read.table(TARGET_TABLE).groupBy("loyalty_tier").count().orderBy("loyalty_tier").show()